# Delta Table - All Data Types Demo
Creates a Delta table demonstrating all major Spark data types including **TIMESTAMP** (UTC-normalized) and **TIMESTAMP_NTZ** (no timezone), with full microsecond precision.

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType,
    ShortType, ByteType, FloatType, DoubleType, DecimalType,
    BooleanType, DateType, TimestampType, TimestampNTZType,
    BinaryType, ArrayType, MapType
)
from decimal import Decimal
import random
import string
import datetime

In [ ]:
# Define schema with a wide range of data types

schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("byte_col", ByteType(), True),
    StructField("short_col", ShortType(), True),
    StructField("int_col", IntegerType(), True),
    StructField("long_col", LongType(), True),
    StructField("float_col", FloatType(), True),
    StructField("double_col", DoubleType(), True),
    StructField("decimal_col", DecimalType(38, 18), True),
    StructField("string_col", StringType(), True),
    StructField("boolean_col", BooleanType(), True),
    StructField("date_col", DateType(), True),
    StructField("timestamp_col", TimestampType(), True),
    StructField("timestamp_ntz_col", TimestampNTZType(), True),
    StructField("binary_col", BinaryType(), True),
    StructField("array_col", ArrayType(IntegerType()), True),
    StructField("map_col", MapType(StringType(), IntegerType()), True),
])

In [ ]:
# Generate 100 rows of randomized data with full microsecond precision

random.seed(42)

def random_string(length=10):
    return ''.join(random.choices(string.ascii_letters + string.digits, k=length))

def random_timestamp():
    """Generate a random timestamp with full microsecond precision (6 decimal places)."""
    year = random.randint(2020, 2026)
    month = random.randint(1, 12)
    day = random.randint(1, 28)
    hour = random.randint(0, 23)
    minute = random.randint(0, 59)
    second = random.randint(0, 59)
    microsecond = random.randint(0, 999999)  # Full microsecond precision
    return datetime.datetime(year, month, day, hour, minute, second, microsecond)

def random_binary():
    return bytes(random.getrandbits(8) for _ in range(random.randint(4, 16)))

rows = []
for i in range(1, 101):
    row = (
        i,                                                          # id
        random.randint(-128, 127),                                  # byte_col (TINYINT)
        random.randint(-32768, 32767),                              # short_col (SMALLINT)
        random.randint(-2147483648, 2147483647),                    # int_col (INT)
        random.randint(-9223372036854775808, 9223372036854775807),  # long_col (BIGINT)
        random.uniform(-1e6, 1e6),                                 # float_col (FLOAT)
        random.uniform(-1e15, 1e15),                                # double_col (DOUBLE)
        Decimal(format(random.uniform(-1e10, 1e10), '.18f')),       # decimal_col DECIMAL(38,18)
        random_string(random.randint(5, 50)),                       # string_col (STRING)
        random.choice([True, False]),                               # boolean_col (BOOLEAN)
        datetime.date(
            random.randint(2000, 2026),
            random.randint(1, 12),
            random.randint(1, 28)
        ),                                                          # date_col (DATE)
        random_timestamp(),                                         # timestamp_col - microsecond precision
        random_timestamp(),                                         # timestamp_ntz_col - microsecond precision
        random_binary(),                                            # binary_col (BINARY)
        [random.randint(1, 1000) for _ in range(random.randint(1, 5))],  # array_col
        {random_string(3): random.randint(1, 100) for _ in range(random.randint(1, 3))},  # map_col
    )
    rows.append(row)

print(f"Generated {len(rows)} rows")
print(f"Sample timestamp_col (row 1): {rows[0][11]}")
print(f"Sample timestamp_ntz_col (row 1): {rows[0][12]}")
print(f"Microseconds in timestamp_col: {rows[0][11].microsecond}")
print(f"Microseconds in timestamp_ntz_col: {rows[0][12].microsecond}")

In [ ]:
# Create DataFrame and inspect schema

df = spark.createDataFrame(rows, schema=schema)

print("=== Schema ===")
df.printSchema()

print("\n=== Sample Data (first 5 rows, key columns) ===")
df.select(
    "id", "byte_col", "short_col", "int_col", "long_col",
    "float_col", "double_col", "decimal_col", "boolean_col",
    "date_col", "timestamp_col", "timestamp_ntz_col"
).show(5, truncate=False)

In [ ]:
# Write to Delta table

TABLE_NAME = "all_datatypes_demo"

df.write.format("delta").mode("overwrite").saveAsTable(TABLE_NAME)

print(f"Delta table '{TABLE_NAME}' created successfully with {df.count()} rows.")

In [ ]:
# Verify: read back and confirm precision is preserved

print("=== Reading back from Delta table ===\n")

df_read = spark.read.table(TABLE_NAME)

print("Schema from Delta table:")
df_read.printSchema()

print("\n=== TIMESTAMP vs TIMESTAMP_NTZ comparison ===")
print("TIMESTAMP: Stored as UTC, timezone-aware. Spark adjusts for session timezone on read.")
print("TIMESTAMP_NTZ: Stored as-is, no timezone conversion. What you write is what you read.\n")

df_read.select("id", "timestamp_col", "timestamp_ntz_col").show(10, truncate=False)

# Verify microsecond precision is preserved
print("\n=== Verifying microsecond precision ===")
spark.sql(f"""
    SELECT 
        id,
        timestamp_col,
        timestamp_ntz_col,
        date_format(timestamp_col, 'yyyy-MM-dd HH:mm:ss.SSSSSS') as ts_formatted,
        date_format(timestamp_ntz_col, 'yyyy-MM-dd HH:mm:ss.SSSSSS') as ts_ntz_formatted
    FROM {TABLE_NAME}
    LIMIT 10
""").show(truncate=False)

In [ ]:
# Show Delta table metadata

print("=== Delta Table Detail (DESCRIBE EXTENDED) ===\n")
spark.sql(f"DESCRIBE EXTENDED {TABLE_NAME}").show(100, truncate=False)